<a class="anchor" id='import'>
<font color = '#006400'>
    
# **1. Data Integration** </font>
</a>

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **1.1. Import the needed libraries** </font>

In [26]:
import polars as pl

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **1.2. Integrate the datasets into the notebook** </font>

In [27]:
#Import movies.csv dataset

url_data = "/Users/goncalobento/Downloads/ml-32m/movies.csv"

movies = pl.read_csv(
    url_data,
    separator=',',
    has_header=True
)

In [28]:
movies.head()

movieId,title,genres
i64,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""


In [23]:
#Import rating dataset

url_data = "/Users/goncalobento/Downloads/ml-32m/ratings.csv"

ratings = pl.read_csv(
    url_data,
    separator=',',
    has_header=True
)


In [24]:
ratings.head()

userId,movieId,rating,timestamp
i64,i64,f64,i64
1,17,4.0,944249077
1,25,1.0,944250228
1,29,2.0,943230976
1,30,5.0,944249077
1,32,5.0,943228858


In [17]:
#Import links dataset

url_data = "/Users/goncalobento/Downloads/ml-32m/links.csv"

links = pl.read_csv(
    url_data,
    separator=',',
    has_header=True
)

In [18]:
links.head()

movieId,imdbId,tmdbId
i64,i64,i64
1,114709,862
2,113497,8844
3,113228,15602
4,114885,31357
5,113041,11862


In [21]:
#Import tags dataset

url_data = "/Users/goncalobento/Downloads/ml-32m/tags.csv"

tags = pl.read_csv(
    url_data,
    separator=',',
    has_header=True
)


In [22]:
tags.head()

userId,movieId,tag,timestamp
i64,i64,str,i64
22,26479,"""Kevin Kline""",1583038886
22,79592,"""misogyny""",1581476297
22,247150,"""acrophobia""",1622483469
34,2174,"""music""",1249808064
34,2174,"""weird""",1249808102


<a class="anchor" id='import'>
<font color = '#006400'>
    
# **2. Data Access, Exploration and Understanding** </font>
</a>

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.1. Ratings** </font>

In [29]:
ratings.describe()

statistic,userId,movieId,rating,timestamp
str,f64,f64,f64,f64
"""count""",3.2000204e7,3.2000204e7,3.2000204e7,3.2000204e7
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",100278.506411,29318.610122,3.540396,1.2752e9
"""std""",57949.046233,50958.16088,1.058986,2.5616e8
"""min""",1.0,1.0,0.5,7.89652004e8
"""25%""",50053.0,1233.0,3.0,1.0510e9
"""50%""",100297.0,3452.0,3.5,1.2726e9
"""75%""",150451.0,44199.0,4.0,1.5032e9
"""max""",200948.0,292757.0,5.0,1.6972e9


In [30]:
ratings = ratings.unique()

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.2. Movies** </font>

In [29]:
movies.describe()

statistic,movieId,title,genres
str,f64,str,str
"""count""",87585.0,"""87585""","""87585"""
"""null_count""",0.0,"""0""","""0"""
"""mean""",157651.365519,null,null
"""std""",79013.402099,null,null
"""min""",1.0,""" (2019)""","""(no genres listed)"""
"""25%""",112657.0,null,null
"""50%""",165741.0,null,null
"""75%""",213203.0,null,null
"""max""",292757.0,"""줄탁동시 (2012)""","""Western"""


In [49]:
movies.filter(pl.col('movieId') == 29).select('genres')

genres
str
"""Adventure|Drama|Fantasy|Myster…"


In [30]:
movies.filter(pl.col('genres') == 'no genres listed')

movieId,title,genres
i64,str,str


In [17]:
valid_genres = [
    "Action",
    "Adventure",
    "Animation",
    "Children's",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western",
    "(no genres listed)"
]


invalid_genres = (
    movies
    .select(pl.col("genres").str.split('|').explode())
    .filter(~pl.col("genres").is_in(valid_genres))
    .unique()
)

print(invalid_genres)


shape: (2, 1)
┌──────────┐
│ genres   │
│ ---      │
│ str      │
╞══════════╡
│ Children │
│ IMAX     │
└──────────┘


In [33]:
movies.filter(pl.col('genres') == 'IMAX')

movieId,title,genres
i64,str,str


In [32]:
invalid_genres = ["IMAX",  "(no genres listed)"]
movies = movies.filter(~pl.col("genres").is_in(invalid_genres))

In [34]:
movies.describe()

statistic,movieId,title,genres
str,f64,str,str
"""count""",80504.0,"""80504""","""80504"""
"""null_count""",0.0,"""0""","""0"""
"""mean""",155501.531576,null,null
"""std""",81084.239109,null,null
"""min""",1.0,""" (2019)""","""Action"""
"""25%""",104285.0,null,null
"""50%""",164739.0,null,null
"""75%""",213756.0,null,null
"""max""",292757.0,"""貞子3D (2012)""","""Western"""


In [35]:
movies = movies.with_columns(
    pl.col("title").str.replace(r" \(\d{4}\)$", "", literal=False).alias("title")
)

In [36]:
movies.head(1000)

movieId,title,genres
i64,str,str
1,"""Toy Story""","""Adventure|Animation|Children|C…"
2,"""Jumanji""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men""","""Comedy|Romance"""
4,"""Waiting to Exhale""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II""","""Comedy"""
…,…,…
1018,"""That Darn Cat!""","""Children|Comedy|Mystery"""
1019,"""20,000 Leagues Under the Sea""","""Adventure|Drama|Sci-Fi"""
1020,"""Cool Runnings""","""Comedy"""


In [37]:
movies.describe()

statistic,movieId,title,genres
str,f64,str,str
"""count""",80504.0,"""80504""","""80504"""
"""null_count""",0.0,"""0""","""0"""
"""mean""",155501.531576,null,null
"""std""",81084.239109,null,null
"""min""",1.0,"""""","""Action"""
"""25%""",104285.0,null,null
"""50%""",164739.0,null,null
"""75%""",213756.0,null,null
"""max""",292757.0,"""貞子3D""","""Western"""


In [38]:
movies = movies.unique()

In [39]:
movies.describe()

statistic,movieId,title,genres
str,f64,str,str
"""count""",80504.0,"""80504""","""80504"""
"""null_count""",0.0,"""0""","""0"""
"""mean""",155501.531576,null,null
"""std""",81084.239109,null,null
"""min""",1.0,"""""","""Action"""
"""25%""",104285.0,null,null
"""50%""",164739.0,null,null
"""75%""",213756.0,null,null
"""max""",292757.0,"""貞子3D""","""Western"""


In [40]:
movies = movies.with_columns(
    pl.col("genres").str.split("|")
).explode("genres")

# Agora cada linha tem apenas um género
print(movies.head(10))


shape: (10, 3)
┌─────────┬─────────────────────────────────┬─────────────┐
│ movieId ┆ title                           ┆ genres      │
│ ---     ┆ ---                             ┆ ---         │
│ i64     ┆ str                             ┆ str         │
╞═════════╪═════════════════════════════════╪═════════════╡
│ 249978  ┆ Andaz                           ┆ Drama       │
│ 144938  ┆ Money for Nothing: Inside the … ┆ Documentary │
│ 155555  ┆ Maïna                           ┆ Adventure   │
│ 155555  ┆ Maïna                           ┆ Drama       │
│ 246206  ┆ Polle fiction                   ┆ Comedy      │
│ 284567  ┆ Look At Me: XXXTENTACION        ┆ Documentary │
│ 173907  ┆ Le grand Méliès                 ┆ Documentary │
│ 173907  ┆ Le grand Méliès                 ┆ Drama       │
│ 129070  ┆ Cover Girl Models               ┆ Action      │
│ 129070  ┆ Cover Girl Models               ┆ Drama       │
└─────────┴─────────────────────────────────┴─────────────┘


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.3. Links** </font>

In [63]:
links.head()

movieId,imdbId,tmdbId
i64,i64,i64
1,114709,862
2,113497,8844
3,113228,15602
4,114885,31357
5,113041,11862


In [64]:
links.describe()

statistic,movieId,imdbId,tmdbId
str,f64,f64,f64
"""count""",87585.0,87585.0,87461.0
"""null_count""",0.0,0.0,124.0
"""mean""",157651.365519,2.7928e6,241382.280422
"""std""",79013.402099,4.2789e6,247146.667043
"""min""",1.0,1.0,2.0
"""25%""",112657.0,94642.0,46836.0
"""50%""",165741.0,492996.0,139272.0
"""75%""",213203.0,3.877296e6,381693.0
"""max""",292757.0,2.9081098e7,1.186337e6


In [65]:
links = links.unique()

In [66]:
links.describe()

statistic,movieId,imdbId,tmdbId
str,f64,f64,f64
"""count""",87585.0,87585.0,87461.0
"""null_count""",0.0,0.0,124.0
"""mean""",157651.365519,2.7928e6,241382.280422
"""std""",79013.402099,4.2789e6,247146.667043
"""min""",1.0,1.0,2.0
"""25%""",112657.0,94642.0,46836.0
"""50%""",165741.0,492996.0,139272.0
"""75%""",213203.0,3.877296e6,381693.0
"""max""",292757.0,2.9081098e7,1.186337e6


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.4. Tags** </font>

In [67]:
tags.head()

userId,movieId,tag,timestamp
i64,i64,str,i64
22,26479,"""Kevin Kline""",1583038886
22,79592,"""misogyny""",1581476297
22,247150,"""acrophobia""",1622483469
34,2174,"""music""",1249808064
34,2174,"""weird""",1249808102


In [68]:
tags.describe()

statistic,userId,movieId,tag,timestamp
str,f64,f64,str,f64
"""count""",2.000072e6,2.000072e6,"""2000072""",2.000072e6
"""null_count""",0.0,0.0,"""0""",0.0
"""mean""",81928.586291,71893.261729,null,1.5289e9
"""std""",38106.498431,74803.79499,null,1.2908e8
"""min""",22.0,1.0,""" The Asylum""",1.1354e9
"""25%""",68413.0,4011.0,null,1.4736e9
"""50%""",78213.0,52328.0,null,1.5741e9
"""75%""",103698.0,122294.0,null,1.6147e9
"""max""",162279.0,292629.0,"""카운트다운""",1.6972e9


In [69]:
tags = tags.unique()

In [70]:
tags.describe()

statistic,userId,movieId,tag,timestamp
str,f64,f64,str,f64
"""count""",2.000072e6,2.000072e6,"""2000072""",2.000072e6
"""null_count""",0.0,0.0,"""0""",0.0
"""mean""",81928.586291,71893.261729,null,1.5289e9
"""std""",38106.498431,74803.79499,null,1.2908e8
"""min""",22.0,1.0,""" The Asylum""",1.1354e9
"""25%""",68413.0,4011.0,null,1.4736e9
"""50%""",78213.0,52328.0,null,1.5741e9
"""75%""",103698.0,122294.0,null,1.6147e9
"""max""",162279.0,292629.0,"""카운트다운""",1.6972e9


<a class="anchor" id='import'>
<font color = '#006400'>
    
# **3. Convert to Parquet** </font>
</a>

In [ ]:
# Define paths
output_dir = "~/Downloads"

# Make sure the directory exists
import os
os.makedirs(output_dir, exist_ok=True)

# Convert and save
ratings.write_parquet(os.path.join(output_dir, "ratings_32M.parquet"))
movies.write_parquet(os.path.join(output_dir, "movies_32M.parquet"))
links.write_parquet(os.path.join(output_dir, "links_32M.parquet"))
tags.write_parquet(os.path.join(output_dir, "tags_32M.parquet"))

In [41]:
output_dir = "~/Downloads"

# Make sure the directory exists
import os
os.makedirs(output_dir, exist_ok=True)
movies.write_parquet(os.path.join(output_dir, "movies_32M_preprocessed.parquet"))

In [22]:
movies.describe()

statistic,movieId,title,genres
str,f64,str,str
"""count""",146769.0,"""146769""","""146769"""
"""null_count""",0.0,"""0""","""0"""
"""mean""",150084.120502,null,null
"""std""",82616.308524,null,null
"""min""",1.0,"""""","""Action"""
"""25%""",93502.0,null,null
"""50%""",158623.0,null,null
"""75%""",210669.0,null,null
"""max""",292757.0,"""貞子3D""","""Western"""


In [25]:
movies.head(1000)

movieId,title,genres
i64,str,str
1,"""Toy Story""","""Adventure"""
1,"""Toy Story""","""Animation"""
1,"""Toy Story""","""Children"""
1,"""Toy Story""","""Comedy"""
1,"""Toy Story""","""Fantasy"""
…,…,…
484,"""Lassie""","""Adventure"""
484,"""Lassie""","""Children"""
485,"""Last Action Hero""","""Action"""
